### SQL Using Python

In [0]:
df_shipped = spark.sql("select * from ecommerce.bronze.orders where order_status = 'shipped'")
display(df_shipped)

# spark.sql("""
#           create or replace table ecommerce.silver.orders 
#           as select * from ecommerce.bronze.orders 
#           where order_status = 'shipped'
#           """).show()

### SQL Using %sql

In [0]:
%sql
select * from ecommerce.bronze.orders where order_status = 'delivered';

-- create or replace table ecommerce.silver.orders 
-- as select * from ecommerce.bronze.orders 
-- where order_status = 'shipped';

### Joins

In [0]:
from pyspark.sql import functions as F, types as T

rows_customers = [
    (1, "Asha", "IN", True),
    (2, "James", "US", False),
    (3, "Leila", "CN", True),
    (4, "Ramesh", "US", None),
    (None, "Chloe", "UK", False)
]

rows_orders = [
    (101, 1, 120.0, "IN"),
    (102, 1, 80.0, "IN"),
    (103, 2, 50.0, "US"),
    (104, 5, 30.0, "DE"),
    (105, 3, 200.0, "CN"),
    (106, None, 15.0, "UK"),
    (107, 3, 40.0, "CN"),
    (108, 2, 75.0, "US"),
]

schema_customers = T.StructType([
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("name", T.StringType(), True),
    T.StructField("country", T.StringType(), True),
    T.StructField("vip", T.BooleanType(), True)
])

schema_orders = T.StructType([
    T.StructField("order_id", T.IntegerType(), True),
    T.StructField("customer_id", T.IntegerType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("country", T.StringType(), True)
])

df_customers = spark.createDataFrame(rows_customers, schema_customers)
df_orders = spark.createDataFrame(rows_orders, schema_orders)

display(df_customers)
display(df_orders)

In [0]:
df_inner = df_orders.join(df_customers, on = "customer_id", how = "inner") # joins wherever the customer_id is the same and present in both tables
df_left = df_orders.join(df_customers, on = "customer_id", how = "left") # keeps all the rows from the left table and only those from the right table where the customer_id is the same
df_right = df_orders.join(df_customers, on = "customer_id", how = "right") # same but with the right table
df_multi = df_orders.join(df_customers, on = ["customer_id", "country"], how = "inner") # joins wherever the customer_id and country are the same and present in both tables

o = df_orders.alias("o")
c = df_customers.alias("c")

df_inner_clean = (
    o.join(c, on="customer_id", how="inner")
    .select(
        "order_id",
        "customer_id",
        "amount",
        F.col("o.country").alias("order_country"),
        "name",
        F.col("c.country").alias("customer_country"),
        "vip"
    )
)

display(df_inner_clean)

Think of it this way: the four types differ on two axes — does it store data physically? and when does it update?

Regular Delta Table
A physical table you write to explicitly. It only changes when your code runs and writes to it. Nothing happens automatically.
Real world example: your seller_performance scorecard. It's a complex multi-step calculation — you want to control exactly when it refreshes, validate it before exposing it, and have it stable for reporting. You run it on a schedule every night at midnight. Between runs, the numbers don't change.

View
No physical storage — it's a saved SQL query that runs live every time someone queries it. Fast to create, zero maintenance, but every query re-executes the underlying joins and aggregations from scratch.
Real world example: a simple lookup like "give me all delivered orders with their customer city." If an analyst queries it 100 times, the full join runs 100 times. Fine for low-traffic, simple queries. Bad for expensive aggregations or high-traffic dashboards.

Materialized View
Like a view, but Databricks physically stores the result and automatically refreshes it when the upstream tables change. You get the freshness of a view without re-running the query on every read.
Real world example: monthly_revenue by product category. The calculation is expensive — joining orders, items, and products across 100k rows. You want it to stay current as Silver data updates, but you don't want every dashboard load to re-run the full aggregation. Databricks handles the refresh for you.
The key difference from a regular table: you don't write to it. You define it once and Databricks keeps it current.

Streaming Table
Continuously ingests new rows as they arrive rather than waiting for a batch run. Built on Lakeflow Declarative Pipelines, it processes only the new data since the last run — not the full table every time.
Real world example: your live_orders feed showing active orders in processing or shipped status. Operations needs this updated continuously, not once a night. As new orders come in from Silver, they appear in the streaming table within minutes. Old entries that transition to delivered get handled by the pipeline logic.
The key difference from the others: it's designed for append or change streams, not for re-computing from scratch.

The one-sentence decision rule:

Need to control exactly when it updates → regular table
Simple query, low traffic, don't want to manage storage → view
Expensive aggregation that should stay current automatically → materialized view
Continuously arriving data that needs near-real-time freshness → streaming table